# Tick Detection - YOLOv8 Training\nTrain a model to detect ticks (клещи) in photos.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install ultralytics
!pip install ultralytics -q

In [ ]:
import os
# Upload dataset to Google Drive first, then set this path
# Copy dataset from Drive to Colab (faster I/O)
DRIVE_DATASET = '/content/drive/MyDrive/tick_dataset'  # <-- CHANGE THIS
!cp -r "$DRIVE_DATASET" /content/dataset
!ls /content/dataset

## 2. Train YOLOv8

In [ ]:
from ultralytics import YOLO
import torch

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = 0
else:
    device = 'cpu'

# Update data.yaml paths for Colab
import yaml
data_path = '/content/dataset/data.yaml'
with open(data_path) as f:
    data = yaml.safe_load(f)
data['train'] = '/content/dataset/images/train'
data['val'] = '/content/dataset/images/val'
with open(data_path, 'w') as f:
    yaml.dump(data, f)
print('data.yaml updated for Colab')

In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data=data_path,
    epochs=150,
    imgsz=640,
    batch=16,
    device=device,
    patience=30,
    save=True,
    project='/content/drive/MyDrive/tick_dataset/runs',
    name='tick_detector',
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    seed=42,
)

In [ ]:
# Show results
print(f"Best model: {results.save_dir / 'weights' / 'best.pt'}")
print(f"Last model: {results.save_dir / 'weights' / 'last.pt'}")

## 3. Export model for inference

In [ ]:
# Export to ONNX (runs faster on CPU)
best_path = results.save_dir / 'weights' / 'best.pt'
model.export(format='onnx', imgsz=640)
print(f"ONNX model: {best_path.with_suffix('.onnx')}")

## 4. Test on sample images

In [ ]:
import glob
val_imgs = glob.glob('/content/dataset/images/val/*.jpg') + glob.glob('/content/dataset/images/val/*.png')
results = model(val_imgs[:5])  # test first 5
for r in results:
    print(f"{r.path}: {len(r.boxes)} tick(s) detected")

In [ ]:
# Display results with boxes
from IPython.display import display
for r in results[:3]:
    display(r.plot()[:, :, ::-1])  # BGR->RGB